# QuizForge - fine-tune Qwen 2.5 (3B and 7B) for MCQ generation

This notebook fine-tunes both models for exactly the task the app gives them: *write multiple-choice questions as JSON
from source material*. It uses the app's own system prompt, prompt builder and validator, so the training data matches
what the app sends and accepts.

**Steps** (run the cells in order, about 2-3 hours on a free T4 GPU):
1. Get the project code and install Unsloth.
2. Download SciQ (a public science-MCQ dataset with source paragraphs) and convert it to the app's format.
3. *Teacher data*: run the app's real generator with `qwen2.5:7b` (through Ollama) on held-out SciQ passages and keep only
   questions the validator accepts **and** `qwen2.5:3b` answers the same way blind.
4. Build `train.jsonl` / `val.jsonl`.
5. QLoRA fine-tuning of the 3B model, then of the 7B model; each is exported as a q4_K_M GGUF file.
6. Save the GGUF files to Google Drive, download them, and run `finetune\import_finetuned.bat` on your laptop.

**Before you start:** *Runtime -> Change runtime type -> T4 GPU*. If the session disconnects, the outputs saved to Drive
in each step survive; re-run the setup cells and continue from the step you were at.

Data licence: SciQ is CC BY-NC 3.0 (non-commercial use). Base models: Qwen2.5-3B-Instruct (Qwen Research Licence,
non-commercial) and Qwen2.5-7B-Instruct (Apache 2.0).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Project code and packages
Option A clones the GitHub repository (it must contain the `finetune/` folder).
Option B: run `.venv\Scripts\python.exe -m finetune.make_colab_bundle` on your laptop and upload `quizforge_colab.zip`
with the file browser on the left, then set `USE_ZIP = True`.

In [ ]:
import os, subprocess, zipfile
USE_ZIP = False
REPO = "https://github.com/Suketu-ADT/quizforge-mcq-generator.git"
if USE_ZIP:
    zipfile.ZipFile("/content/quizforge_colab.zip").extractall("/content/quizforge")
elif not os.path.exists("/content/quizforge"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/quizforge"], check=True)
os.chdir("/content/quizforge")
assert os.path.exists("finetune/train.py"), "finetune/ is missing - push it to GitHub or use the zip (option B)"
print("project ready:", os.getcwd())

In [ ]:
!pip -q install unsloth
!pip -q install --upgrade datasets

In [ ]:
# Keep outputs on Google Drive so a disconnect does not lose them
SAVE_TO_DRIVE = True
OUT = "/content/quizforge/finetune"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/quizforge_finetune"
    os.makedirs(DRIVE, exist_ok=True)
    print("Drive folder:", DRIVE)

def save(path):
    if SAVE_TO_DRIVE:
        import shutil
        shutil.copy(path, DRIVE)
        print("saved to Drive:", os.path.basename(path))

## 2. SciQ
11,679 training rows: a question, the correct answer, three distractors and the paragraph that supports it.
The training split becomes SciQ examples; the validation split only supplies passages for the teacher step, so the two
never share a paragraph.

In [ ]:
import json
from datasets import load_dataset
sciq = load_dataset("allenai/sciq")
os.makedirs(f"{OUT}/data", exist_ok=True)
for split in ("train", "validation"):
    with open(f"{OUT}/data/sciq_{split}.jsonl", "w", encoding="utf-8") as f:
        for row in sciq[split]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(split, len(sciq[split]))

## 3. Teacher data (optional but recommended, ~30-45 min)
Installs Ollama inside this Colab machine, pulls the two base models and runs `finetune/distill.py`.
Set `TEACHER_PASSAGES = 0` to skip this step.

In [ ]:
TEACHER_PASSAGES = 80
if TEACHER_PASSAGES:
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    import subprocess, time, requests
    server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/version", timeout=2); break
        except Exception:
            time.sleep(1)
    !ollama pull qwen2.5:7b
    !ollama pull qwen2.5:3b

In [ ]:
if TEACHER_PASSAGES:
    !python -m finetune.distill --rows finetune/data/sciq_validation.jsonl --out finetune/data/distilled.jsonl --passages {TEACHER_PASSAGES}
    save(f"{OUT}/data/distilled.jsonl")
    server.terminate()                      # free the GPU for training
    !pkill -f "ollama" || true

## 4. Training data

In [ ]:
!python -m finetune.build_dataset --sciq finetune/data/sciq_train.jsonl --distilled finetune/data/distilled.jsonl --out finetune/data --sciq-examples 1800
for f in ("train.jsonl", "val.jsonl", "dataset_info.json"):
    save(f"{OUT}/data/{f}")

In [ ]:
# look at one training example
ex = json.loads(open(f"{OUT}/data/train.jsonl", encoding="utf-8").readline())
print(ex["messages"][1]["content"][-700:])
print("\n--- target ---\n", ex["messages"][2]["content"][:900])

## 5a. Fine-tune the 3B model (~20-30 min incl. GGUF export)

In [ ]:
!python -m finetune.train --base unsloth/Qwen2.5-3B-Instruct-bnb-4bit --name quizforge-3b --data finetune/data --out finetune/exports
save(f"{OUT}/exports/quizforge-3b-report.json")
save(f"{OUT}/exports/quizforge-3b.Q4_K_M.gguf")

## 5b. Fine-tune the 7B model (~45-70 min incl. GGUF export)
Uses batch size 1 with 8 accumulation steps to fit the T4's 16 GB.

In [ ]:
!python -m finetune.train --base unsloth/Qwen2.5-7B-Instruct-bnb-4bit --name quizforge-7b --data finetune/data --out finetune/exports --batch 1 --grad-accum 8
save(f"{OUT}/exports/quizforge-7b-report.json")
save(f"{OUT}/exports/quizforge-7b.Q4_K_M.gguf")

## 6. Results and download
The reports show the validation loss before and after training and the format check (how many fine-tuned replies pass
the app's validator **without** a JSON-schema constraint).

Download the two `.gguf` files from the Drive folder `quizforge_finetune` (more reliable than the browser for 2-5 GB),
put them in `finetune\exports\` on your laptop and double-click `finetune\import_finetuned.bat`.
Then `finetune\compare_finetuned.bat` compares each fine-tuned model with its base model on all three tests.

In [ ]:
for name in ("quizforge-3b", "quizforge-7b"):
    p = f"{OUT}/exports/{name}-report.json"
    if os.path.exists(p):
        r = json.load(open(p))
        print(name, {k: r.get(k) for k in ("train_examples", "train_loss", "val_loss_before", "val_loss_after", "gguf_gb")})
        print("  format check:", r.get("format_check"))